# TripMind request classifier (TF-IDF + logistic regression)

Sorts a traveller's support message into a category (cancellation or refund, change a booking,
baggage and seats, ...) to route support tickets and help the assistant.

**Data:** Bitext *travel* customer-support dataset on Hugging Face
(`bitext/Bitext-travel-llm-chatbot-training-dataset`, about 31,700 rows, 33 intents,
licence CDLA-Sharing-1.0). Bitext generated it with their language tools, so its phrasing is
cleaner than real messages and the score on its own test split is optimistic. The notebook
also reports accuracy on hand-written messages in TripMind's own style.

**Run:** anywhere with internet (the CSV downloads from Hugging Face): Kaggle with internet
on, or locally from `backend/`:
`uv run --with nbconvert --with ipykernel jupyter nbconvert --to notebook --execute --inplace ../models/request_classifier/request_classifier.ipynb`

**Hand back:** `model.json` and `model_card.json` in `models/request_classifier/artifacts/`
(written there directly when run locally).

In [ ]:
import os
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split

URL = ("https://huggingface.co/datasets/bitext/Bitext-travel-llm-chatbot-training-dataset/"
       "resolve/main/bitext-travel-llm-chatbot-training-dataset.csv")
df = pd.read_csv(URL, usecols=["instruction", "intent", "category"])
print(df.shape, df.intent.nunique(), "intents")

Intents are grouped into TripMind categories with the mapping the API uses (`backend/app/modules/ml/request_classifier.py`).

In [ ]:
import re

INTENT_CATEGORIES: dict[str, str] = {
    "cancel_flight": "cancellation_refund",
    "cancel_trip": "cancellation_refund",
    "get_refund": "cancellation_refund",
    "check_cancellation_fee": "cancellation_refund",
    "change_flight": "change_booking",
    "change_trip": "change_booking",
    "check_baggage_allowance": "baggage_seats",
    "change_seat": "baggage_seats",
    "choose_seat": "baggage_seats",
    "check_in": "check_in_boarding",
    "get_boarding_pass": "check_in_boarding",
    "print_boarding_pass": "check_in_boarding",
    "check_flight_status": "flight_status",
    "check_arrival_time": "flight_status",
    "check_departure_time": "flight_status",
    "check_flight_insurance_coverage": "insurance",
    "purchase_flight_insurance": "insurance",
    "search_flight_insurance": "insurance",
    "check_trip_insurance_coverage": "insurance",
    "purchase_trip_insurance": "insurance",
    "search_trip_insurance": "insurance",
    "book_flight": "new_booking",
    "book_trip": "new_booking",
    "search_flight": "new_booking",
    "search_trip": "new_booking",
    "check_flight_offers": "new_booking",
    "check_trip_offers": "new_booking",
    "check_flight_prices": "new_booking",
    "check_trip_prices": "new_booking",
    "check_flight_reservation": "existing_booking",
    "check_trip_details": "existing_booking",
    "check_trip_plan": "existing_booking",
    "human_agent": "human_agent",
}

RULES: tuple[tuple[str, re.Pattern[str]], ...] = tuple(
    (category, re.compile(pattern, re.IGNORECASE))
    for category, pattern in (
        ("human_agent", r"\b(human|agent|real person|speak to (someone|a person)|call me)\b"),
        ("cancellation_refund", r"\b(cancel\w*|refund\w*|money back|reimburse\w*)\b"),
        ("change_booking", r"\b(change|reschedul\w*|move|modify|different (date|flight|time))\b"),
        ("baggage_seats", r"\b(bag\w*|luggage|suitcase|kg|seat\w*|legroom|meal)\b"),
        ("check_in_boarding", r"\b(check[- ]?in|boarding pass|boarding)\b"),
        ("flight_status", r"\b(delay\w*|status|on time|depart\w*|arriv\w*|late)\b"),
        ("insurance", r"\binsur\w*\b"),
        ("existing_booking", r"\b(my booking|reservation|itinerary|confirmation|reference)\b"),
        ("new_booking", r"\b(book|price|cheap\w*|fare|offer|deal|search|find)\b"),
    )
)


unmapped = set(df.intent) - set(INTENT_CATEGORIES)
assert not unmapped, f"intents without a category: {unmapped}"
df["label"] = df.intent.map(INTENT_CATEGORIES)
print(df.label.value_counts().to_dict())

In [ ]:
train, test = train_test_split(df, test_size=0.2, random_state=7, stratify=df.label)
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=20_000, sublinear_tf=True)
model = LogisticRegression(C=8.0, max_iter=3000, class_weight="balanced")
model.fit(vectorizer.fit_transform(train.instruction), train.label)
predicted = model.predict(vectorizer.transform(test.instruction))
print(classification_report(test.label, predicted, digits=3))

**Baseline:** TripMind's keyword rules. **Reality check:** hand-written messages the model never saw.

In [ ]:
def rules(text):
    return next((c for c, p in RULES if p.search(text)), "other")

HANDWRITTEN = [
    ("my flight got cancelled can i get the money back", "cancellation_refund"),
    ("please refund the hotel, we are not going anymore", "cancellation_refund"),
    ("can you push my flight to next friday", "change_booking"),
    ("I need to fly a day later than booked", "change_booking"),
    ("how much luggage is included in my ticket", "baggage_seats"),
    ("I would like a window seat for my daughter", "baggage_seats"),
    ("where can I download my boarding pass", "check_in_boarding"),
    ("when does online check in open", "check_in_boarding"),
    ("is my flight to goa on time today", "flight_status"),
    ("what time do we land in mumbai", "flight_status"),
    ("does my booking include travel insurance", "insurance"),
    ("I want to buy cover for my trip", "insurance"),
    ("show me cheap flights to bangalore next week", "new_booking"),
    ("book me a flight and a hotel in jaipur", "new_booking"),
    ("can you check the details of my reservation", "existing_booking"),
    ("what is in my itinerary for december", "existing_booking"),
    ("let me talk to a human", "human_agent"),
    ("I want to speak with customer service", "human_agent"),
]
hand_text, hand_label = zip(*HANDWRITTEN)
metrics = {
    "macro_f1": f1_score(test.label, predicted, average="macro"),
    "accuracy": accuracy_score(test.label, predicted),
    "baseline_accuracy": accuracy_score(test.label, [rules(t) for t in test.instruction]),
    "handwritten_accuracy": accuracy_score(hand_label, model.predict(vectorizer.transform(hand_text))),
    "handwritten_baseline_accuracy": accuracy_score(hand_label, [rules(t) for t in hand_text]),
    "test_rows": float(len(test)),
}
print(metrics)
assert metrics["accuracy"] > metrics["baseline_accuracy"], "model does not beat the rules"

In [ ]:
from __future__ import annotations

import datetime
import json
from pathlib import Path
from typing import Any

FORMAT = "tfidf-logreg-v1"


def export_tfidf_logreg(
    vectorizer: Any,
    model: Any,
    out_dir: str | Path,
    *,
    name: str,
    dataset: str,
    metrics: dict[str, float],
    notes: str = "",
    decimals: int = 6,
) -> dict[str, Any]:
    params = vectorizer.get_params()
    unsupported = {
        "analyzer": params["analyzer"] != "word",
        "tokenizer": params["tokenizer"] is not None,
        "preprocessor": params["preprocessor"] is not None,
        "stop_words": params["stop_words"] is not None,
        "strip_accents": params["strip_accents"] is not None,
        "norm": params["norm"] not in ("l2", None),
        "use_idf": not params["use_idf"],
        "binary": params["binary"],
    }
    if any(unsupported.values()):
        raise ValueError(
            f"unsupported vectorizer settings: {[k for k, v in unsupported.items() if v]}"
        )
    vocabulary = {term: int(index) for term, index in vectorizer.vocabulary_.items()}
    payload = {
        "format": FORMAT,
        "lowercase": bool(params["lowercase"]),
        "token_pattern": params["token_pattern"],
        "ngram_range": list(params["ngram_range"]),
        "sublinear_tf": bool(params["sublinear_tf"]),
        "norm": params["norm"],
        "vocabulary": vocabulary,
        "idf": [round(float(v), decimals) for v in vectorizer.idf_],
        "classes": [str(c) for c in model.classes_],
        "coef": [[round(float(v), decimals) for v in row] for row in model.coef_],
        "intercept": [round(float(v), decimals) for v in model.intercept_],
    }
    import sklearn

    card = {
        "name": name,
        "version": datetime.date.today().strftime("%Y.%m.%d"),
        "trained_at": datetime.datetime.now(datetime.UTC).isoformat(),
        "framework": "scikit-learn",
        "framework_version": sklearn.__version__,
        "dataset": dataset,
        "features": [FORMAT],
        "metrics": {k: float(v) for k, v in metrics.items()},
        "notes": notes,
    }
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    (out / "model.json").write_text(json.dumps(payload, separators=(",", ":")), encoding="utf-8")
    (out / "model_card.json").write_text(json.dumps(card, indent=2), encoding="utf-8")
    return card


out = "/kaggle/working" if os.path.isdir("/kaggle/working") else "artifacts"
card = export_tfidf_logreg(
    vectorizer, model, out, name="request_classifier",
    dataset="Bitext travel customer-support dataset (bitext/Bitext-travel-llm-chatbot-training-dataset, CDLA-Sharing-1.0)",
    metrics=metrics, notes="33 Bitext intents grouped into TripMind categories; 80/20 stratified split, seed 7. Synthetic source data.",
)
print(card)